# 05 CNN Image Classification

##  Importing Libraries

We import the libraries and choose the device. If a GPU is available, PyTorch uses it, otherwise it uses the CPU. We also fix the random seed so that the results are the same every time you run the notebook (on the same computer).

In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

**Code Explanation:** `datasets` has the ready-made datasets and `transforms` has the image preprocessing and augmentation steps. `confusion_matrix` and `classification_report` from Scikit-learn are used at the end for the error analysis. `torch.manual_seed(0)` and `np.random.seed(0)` fix the random numbers. The variable `device` is used later to move the model and the data to the GPU when there is one.

**Interpretation:** The printed device is `cuda` when a GPU is used and `cpu` otherwise. Everything in this notebook works on a CPU. The Fashion-MNIST part is fast on a CPU, while the CIFAR-10 part takes noticeably longer. If it is too slow, reduce the `epochs` value in the CIFAR-10 training cell.

# Part 1: Baseline with Fashion-MNIST

Fashion-MNIST has 70,000 grayscale pictures (28 x 28 pixels) of 10 kinds of clothes and shoes. It has the same size and format as the MNIST digits, but it is a little harder, so the differences between models are easier to see.

## 1.1 Dataset Loading

**Definition:** Dataset loading means getting the images and their labels into the program. `torchvision.datasets` downloads a dataset once, stores it in a folder, and gives us an object that we can index like a list.

**Example:** `datasets.FashionMNIST(root="data", train=True, download=True)` downloads the 60,000 training images into a folder called `data`, and `train=False` gives the 10,000 test images.

**Why it is used?** A project must start from a standard, trusted copy of the data. Using the official split into training and test sets makes our results comparable with the results of other people.

In [ ]:
train_raw = datasets.FashionMNIST(root="data", train=True, download=True)
test_raw = datasets.FashionMNIST(root="data", train=False, download=True)
class_names = train_raw.classes
print("Training images:", len(train_raw))
print("Test images:", len(test_raw))
print("Classes:", class_names)

**Code Explanation:** We create two dataset objects, one for the training images and one for the test images. We did not give any `transform`, so the images are returned as they are (as PIL pictures). `classes` is the list of the class names, in the order of the numeric labels 0 to 9.

**Interpretation:** You should see 60000 training images and 10000 test images. The list of classes shows the meaning of the labels: label 0 is `T-shirt/top`, label 1 is `Trouser`, and so on up to label 9, `Ankle boot`. The model works only with the numbers 0 to 9, and we use this list to turn them back into names. The test images are kept aside and used only at the end.

## 1.2 Dataset Inspection

**Definition:** Inspection means checking the basic facts of the data before building anything: the number of images, the image size, the number of channels, the range of the pixel values, the data type and the labels.

**Example:** For one image we check: Is it 28 x 28? Is it grayscale or colour? Do the pixels go from 0 to 255? Which label does it have?

**Why it is used?** Most mistakes in a project come from wrong assumptions about the data, for example a wrong input shape for the first layer, or pixel values that are not scaled. A short inspection finds them early.

In [ ]:
image, label = train_raw[0]
print("One sample type:", type(image).__name__, "| image size:", image.size, "| mode:", image.mode)
print("Label of the first image:", label, "->", class_names[label])
print("Data array shape:", tuple(train_raw.data.shape))
print("Data type:", train_raw.data.dtype)
print("Pixel value range:", train_raw.data.min().item(), "to", train_raw.data.max().item())
train_labels = torch.as_tensor(train_raw.targets).numpy()
print("Images per class:", np.bincount(train_labels))

**Code Explanation:** `train_raw[0]` gives the first image and its label. We print the type, size and mode of the image, then the shape, data type and pixel range of the whole data array, and finally the number of training images in each class with `np.bincount`.

**Interpretation:** The image size `(28, 28)` and the mode `L` mean that every picture is 28 x 28 with a single grayscale channel. The whole array has the shape `(60000, 28, 28)` and the pixels are whole numbers (`uint8`) from 0 to 255. So before using them we must scale them and add a channel dimension, which `ToTensor()` does for us later. The last line shows how many training images each class has. We use it in the next topic.

## 1.3 Class Analysis

**Definition:** Class analysis means looking at how the images are shared among the classes, and at which classes look alike. A dataset is balanced when every class has about the same number of images, and imbalanced when some classes have many more images than others.

**Example:** In a medical dataset with 95 healthy images and 5 sick images, a model that always says "healthy" gets 95 percent accuracy but is useless. The class counts show such a problem.

**Why it is used?** Class balance decides which metric we can trust. With balanced classes, plain accuracy is a fair measure. With imbalanced classes we also need per-class results, and sometimes class weights or resampling.

**Interpretation:** Look at the counts printed in the previous topic. Fashion-MNIST is perfectly balanced: every class has exactly 6,000 training images (and 1,000 test images), so accuracy is a fair measure here and we do not need any special handling. Balance does not mean that all classes are equally easy. By looking at the pictures we expect `T-shirt/top`, `Shirt`, `Pullover` and `Coat` to be confused with each other, because they look similar, and `Sandal`, `Sneaker` and `Ankle boot` to be confused with each other. We check this at the end in the error analysis. If you replace Fashion-MNIST with MNIST, the classes are only roughly balanced, so look at your own counts.

## 1.4 Image Visualization

**Definition:** Visualization means looking at real images of the dataset with their labels.

**Example:** We show the first training image of every class in a grid of 2 rows and 5 columns.

**Why it is used?** Numbers like shapes and counts do not show whether the images are clean, how big the objects are, or how similar the classes look. Looking at the pictures gives us ideas for preprocessing and augmentation.

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(10, 4.5))
for i, a in enumerate(ax.flat):
    index = int(np.where(train_labels == i)[0][0])
    a.imshow(train_raw[index][0], cmap="gray")
    a.set_title(class_names[i])
    a.axis("off")
plt.show()

**Code Explanation:** `ax.flat` lets us go through the 10 cells of the grid with one loop. For every class `i`, `np.where(targets == i)` finds the positions of that class and `[0][0]` takes the first one. `imshow` draws the picture and `cmap="gray"` shows it in grey shades.

**Interpretation:** The pictures are small and low in detail, and each object is centred and fills most of the frame. This tells us that heavy cropping or zooming is not needed. We can also see why some classes will be confused: for example the `Shirt`, the `T-shirt/top` and the `Pullover` share the same overall shape. Because the objects are placed in the same way in every picture, small shifts are the main kind of extra variation that augmentation can add.

## 1.5 Preprocessing

**Definition:** Preprocessing prepares the images for the network. Here it has three parts: (1) `ToTensor()` changes a picture into a tensor of shape (1, 28, 28) with values between 0 and 1, (2) `Normalize(mean, std)` standardizes the pixels with `(pixel - mean) / std`, and (3) we split the 60,000 training images into 54,000 for training and 6,000 for validation.

**Example:** A pixel with the value 255 becomes 1.0 after `ToTensor()`. If the dataset mean is 0.29 and the std is 0.35, it then becomes (1.0 - 0.29) / 0.35, about 2.0.

**Why it is used?** Small, centred numbers make training faster and more stable. The mean and the std must be calculated from the training images only. If we used the validation or the test images, information from them would leak into the training.

**Interpretation:** There is no separate code cell for this topic. The code is in the DataLoader topic (1.7), where the mean and std are computed from the 54,000 training images and then used in the `Normalize` step for the training, validation and test images, all in the same way. The only difference between the training pipeline and the others is the augmentation, which is the next topic. We do not resize the images, because all of them already have the same size.

## 1.6 Augmentation

**Definition:** Augmentation randomly changes the training images a little each time they are used, so that the model sees more variety. It is applied to the training images only. The validation and test images are never augmented, so that they measure the model in a fair and repeatable way.

**Example:** For Fashion-MNIST we use two augmentations. `RandomHorizontalFlip()` mirrors a picture left to right half of the time (a mirrored sneaker is still a sneaker). `RandomCrop(28, padding=2)` adds a border of 2 pixels and cuts a random 28 x 28 window, which moves the object by up to 2 pixels.

**Why it is used?** It reduces overfitting and makes the model less sensitive to the exact position of the object. We choose only the changes that keep the label true: we do not use a vertical flip, because an upside-down shoe or dress is not a normal picture of its class.

**Interpretation:** This topic has no code of its own. The two augmentations are written in the DataLoader topic (1.7), inside `train_transform`. When you compare the training and the validation curves later, remember that the training accuracy is measured on augmented, harder images, so it can be a little lower than the validation accuracy, especially in the first epochs. This is normal and does not mean that something is wrong. If you want to see the effect of augmentation by itself, remove the two augmentation lines and train again.

## 1.7 DataLoader Creation

**Definition:** A `DataLoader` takes a dataset and delivers it in small groups called batches. It can also shuffle the data every epoch. Before that we need three datasets (training, validation, test), each with the right transform.

**Example:** With `batch_size=128` the model receives 128 images at a time, as a tensor of shape (128, 1, 28, 28), together with 128 labels.

**Why it is used?** The whole dataset does not fit into the memory of a GPU at once, and updating the weights after each small batch makes training faster and less noisy than using one image at a time or all images at once.

In [ ]:
indices = torch.randperm(len(train_raw))
n_val = len(indices) // 10
train_idx, val_idx = indices[:-n_val], indices[-n_val:]
pixels = torch.as_tensor(train_raw.data)[train_idx].float() / 255
mean, std = pixels.mean().item(), pixels.std().item()
print("Mean and std of the training pixels: %.4f and %.4f" % (mean, std))
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=2),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])
full_train = datasets.FashionMNIST(root="data", train=True, transform=train_transform)
full_val = datasets.FashionMNIST(root="data", train=True, transform=test_transform)
test_set = datasets.FashionMNIST(root="data", train=False, transform=test_transform)
train_set = Subset(full_train, train_idx)
val_set = Subset(full_val, val_idx)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)
images, labels = next(iter(train_loader))
print("Training / validation / test images:", len(train_set), len(val_set), len(test_set))
print("One batch of images:", tuple(images.shape), "| labels:", tuple(labels.shape))
print("Batch values -> smallest: %.2f, largest: %.2f" % (images.min().item(), images.max().item()))

**Code Explanation:** First we shuffle the numbers 0 to 59999 and keep the last 10 percent as validation indices. We compute the mean and the std from the training indices only. Then we create two transforms: `train_transform` with the augmentation, and `test_transform` without it. We create the training images twice (`full_train` and `full_val`), one with each transform, and use `Subset` with the same index lists to take the training part from the first and the validation part from the second. This way the validation images are not augmented, and no image is in both parts. The test set uses `test_transform`. The training loader shuffles the images in every epoch, while the validation and the test loaders do not need to. The last lines take one batch and print its shape and value range.

**Interpretation:** You should see 54000 training, 6000 validation and 10000 test images. A batch has the shape `(128, 1, 28, 28)`: 128 images, 1 channel, 28 x 28 pixels. This is the `(N, C, H, W)` shape that the convolution layers expect. After the normalization the pixel values go below 0 and above 1 (the range is roughly from -0.8 to 2.0), because they are now centred around 0 and measured in standard deviations.

## 1.8 CNN Architecture

**Definition:** The architecture is the list of layers of the network. Our CNN has two convolution blocks, each with Convolution, Batch Normalization, ReLU and Max Pooling, followed by a classifier that has Flatten, a fully connected layer, Dropout and an output layer with 10 scores (one per class).

**Example:** The image (1, 28, 28) becomes 32 feature maps of 14 x 14 after the first block, and 64 feature maps of 7 x 7 after the second block. Flatten turns them into 64 x 7 x 7 = 3136 numbers, and the fully connected layers turn them into 10 scores.

**Why it is used?** The convolution blocks find the patterns in the picture (edges first, then shapes), and the fully connected part uses them to decide the class. Batch normalization makes the training stable, and dropout reduces overfitting.

In [ ]:
fashion_model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 10),
)
x = torch.zeros(1, 1, 28, 28)
for layer in fashion_model.eval():
    x = layer(x)
    print(layer.__class__.__name__.ljust(12), tuple(x.shape))
fashion_params = sum(p.numel() for p in fashion_model.parameters())
print("Learnable numbers:", fashion_params)

**Code Explanation:** `nn.Sequential` joins the layers in order. We pass one empty image through the model, one layer at a time, and print the shape after every layer. `padding=1` keeps the height and width the same in the convolution layers, and `MaxPool2d(2)` halves them. The last line counts all the learnable numbers (weights and biases).

**Interpretation:** Follow the shapes: `(1, 32, 28, 28)` after the first convolution, `(1, 32, 14, 14)` after pooling, `(1, 64, 7, 7)` after the second block, `(1, 3136)` after Flatten, and finally `(1, 10)`. The image gets smaller while the number of feature maps grows. The final 10 numbers are raw scores (logits), not probabilities. We do not add a Softmax layer, because the loss function `cross_entropy` applies it inside. Most of the learnable numbers sit in the first fully connected layer (3136 x 128), not in the convolution layers.

## 1.9 Training

**Definition:** Training means showing the training images to the network batch by batch and changing its weights to reduce the loss. For every batch we do four steps: (1) the forward pass gives the scores, (2) the loss measures the error with `cross_entropy`, (3) `loss.backward()` finds how each weight should change, and (4) `optimizer.step()` changes the weights. One pass over all the training images is one epoch.

**Example:** With 54,000 images and a batch size of 128, one epoch has about 422 weight updates. We train for 8 epochs with the Adam optimizer.

**Why it is used?** Training is how the kernels and weights learn useful values. After every epoch we also measure the loss and accuracy on the validation set, to see how the model works on images it has not trained on.

We first define two helper functions. `evaluate` measures the loss and accuracy of a model on a loader. `train_model` trains a model and keeps the best weights, which are the weights of the epoch with the highest validation accuracy.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += F.cross_entropy(scores, labels, reduction="sum").item()
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)
def train_model(model, train_loader, val_loader, epochs, lr=0.001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = 0, None
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            scores = model(images)
            loss = F.cross_entropy(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(total_loss / len(train_loader.dataset))
        history["train_acc"].append(correct / len(train_loader.dataset))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        print("Epoch %2d | train loss %.3f, acc %.3f | val loss %.3f, acc %.3f" % (
            epoch + 1, history["train_loss"][-1], history["train_acc"][-1], val_loss, val_acc))
    model.load_state_dict(best_state)
    print("Best validation accuracy: %.3f at epoch %d" % (best_acc, history["val_acc"].index(best_acc) + 1))
    return history

**Code Explanation:** `evaluate` switches the model to evaluation mode with `model.eval()` (dropout off, batch normalization uses its saved averages) and uses `torch.no_grad()` because no weights are changed. It adds up the loss and the number of correct predictions over all batches and returns their averages. `train_model` switches to `model.train()`, goes through the training batches, and does the four steps (forward, loss, backward, update). `optimizer.zero_grad()` clears the old gradients before each new batch. After each epoch it calls `evaluate` on the validation loader and saves the numbers in `history`. Whenever the validation accuracy is the best so far, it saves a copy of the weights. At the end it loads those best weights back into the model.

**Interpretation:** Nothing is printed by this cell, because it only defines the functions. The training accuracy printed later is the average over the augmented batches of the epoch while the weights are still changing, so it is not exactly the accuracy of the final model on clean images.

In [ ]:
epochs = 8
start = time.time()
fashion_history = train_model(fashion_model, train_loader, val_loader, epochs)
fashion_time = time.time() - start
print("Training time: %.0f seconds" % fashion_time)

**Code Explanation:** We train the `fashion_model` for 8 epochs and measure the time with `time.time()`. One line is printed for every epoch, with the training loss and accuracy and the validation loss and accuracy.

**Interpretation:** Read the lines from the top to the bottom. The loss should fall quickly in the first epochs, and the accuracy should rise quickly and then flatten. Because of dropout and augmentation, the training accuracy is often a little lower than the validation accuracy in the first epochs. The last line tells you the epoch with the best validation accuracy, and the model keeps the weights of that epoch. If the best epoch is the last one, the model was still improving and more epochs could help.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(fashion_history["train_loss"], label="Training")
ax[0].plot(fashion_history["val_loss"], label="Validation")
ax[0].set_title("Loss")
ax[1].plot(fashion_history["train_acc"], label="Training")
ax[1].plot(fashion_history["val_acc"], label="Validation")
ax[1].set_title("Accuracy")
for a in ax:
    a.set_xlabel("Epoch")
    a.legend()
plt.show()

**Code Explanation:** The left chart shows the loss and the right chart shows the accuracy, each for the training and for the validation images, over the epochs.

**Interpretation:** This is the chart to study. If both losses go down and the two lines stay close together, the model is learning well and is not overfitting. If the training line keeps improving while the validation line stops improving or gets worse, the model is starting to overfit: it memorizes the training images. If both lines are high and flat, the model is underfitting and needs more capacity or more training. A jumpy validation line usually means that the learning rate is a bit high or that the validation set is small.

## 1.10 Validation

**Definition:** Validation means measuring the model during training on images that are not used to change its weights. The validation set is used to make decisions: how many epochs to train, which weights to keep, which architecture or settings to choose.

**Example:** In our code the validation set (6,000 images) is checked after every epoch, and the weights of the epoch with the best validation accuracy are kept. This is a simple form of model selection.

**Why it is used?** The training accuracy cannot tell whether the model works on new images, because the model has seen the training images. The test set must not be used for decisions either, because after many decisions it would no longer be a fair final exam. So we need a third set in between.

**Interpretation:** There is no separate code for this topic: the validation code is inside `evaluate` and `train_model` (1.9), and the validation curves are in the chart above. Three things to remember when you read the validation results. First, the validation accuracy is slightly optimistic for the final model, because we picked the best epoch by looking at it. Second, a gap between the training and validation accuracy is a sign of overfitting, and a validation loss that rises while the training loss falls is the clearest sign. Third, if you change the architecture or the settings many times based on the validation score, the validation score slowly becomes less honest, which is the reason why the test set is kept untouched for the very end.

## 1.11 Testing

**Definition:** Testing means measuring the final model once on the test set, which was never used for training or for decisions. The test accuracy is the number we report as the performance of the model. We also look at the accuracy of every class and at precision, recall and F1-score.

**Example:** If the model classifies 9,200 of the 10,000 test images correctly, the test accuracy is 92 percent. Recall for a class is the share of that class that the model finds, and precision is the share of the images predicted as that class that are really that class.

**Why it is used?** It gives an honest estimate of how the model will work on new images. The per-class results show where the model is strong and where it is weak, which one overall number hides.

In [ ]:
def get_predictions(model, loader):
    model.eval()
    true, pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            scores = model(images.to(device))
            true.append(labels.numpy())
            pred.append(scores.argmax(dim=1).cpu().numpy())
    return np.concatenate(true), np.concatenate(pred)
fashion_test_loss, fashion_test_acc = evaluate(fashion_model, test_loader)
print("Test loss: %.4f | Test accuracy: %.2f percent" % (fashion_test_loss, fashion_test_acc * 100))
true, pred = get_predictions(fashion_model, test_loader)
per_class = pd.Series(pred == true).groupby(true).mean() * 100
per_class.index = class_names
print(per_class.round(1).sort_values())
print(classification_report(true, pred, target_names=class_names, digits=3, zero_division=0))

**Code Explanation:** `get_predictions` runs the model on a loader and returns the true labels and the predicted labels as arrays (the predicted class is the one with the highest score, `argmax`). We then call `evaluate` on the test loader for the overall loss and accuracy. For the accuracy of each class we check where `pred == true`, group the results by the true label and take the mean. `classification_report` from Scikit-learn prints precision, recall and F1-score for every class.

**Interpretation:** Compare the test accuracy with the best validation accuracy from the training. They should be close. A test accuracy much lower than the validation accuracy would mean that the validation score was too optimistic. The per-class list is sorted from the weakest class to the strongest. For Fashion-MNIST models, the weakest classes are usually `Shirt`, and then `T-shirt/top`, `Pullover` and `Coat`, while `Trouser`, `Bag`, `Sneaker` and `Ankle boot` are usually the easiest. Check whether your output shows this. In the report, a low recall means that the model misses many images of that class, and a low precision means that other classes are wrongly given that label. The overall accuracy hides these differences.

## 1.12 Error Analysis

**Definition:** Error analysis means studying the mistakes of the model: which classes are confused with each other, and what the wrongly classified images look like. The main tool is the confusion matrix, a table where each row is the true class, each column is the predicted class, and each cell counts the images.

**Example:** If the cell (row `Shirt`, column `T-shirt/top`) contains 180, then 180 shirts were predicted as T-shirts. The numbers on the diagonal are the correct predictions.

**Why it is used?** The accuracy tells us how often the model is wrong, but not why. Error analysis shows whether the mistakes are reasonable (very similar classes), whether some labels in the data are doubtful, and what to improve next (more data for one class, a different augmentation, a bigger model).

In [ ]:
matrix = confusion_matrix(true, pred)
plt.figure(figsize=(7, 6))
plt.imshow(matrix, cmap="Blues")
plt.xticks(range(10), class_names, rotation=90)
plt.yticks(range(10), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()
pairs = [(class_names[i], class_names[j], matrix[i, j]) for i in range(10) for j in range(10) if i != j]
pairs = sorted(pairs, key=lambda p: p[2], reverse=True)
for true_name, pred_name, count in pairs[:5]:
    print(true_name, "predicted as", pred_name, ":", count, "times")
wrong = np.where(pred != true)[0]
print("Wrong predictions:", len(wrong), "out of", len(true))
fig, ax = plt.subplots(2, 5, figsize=(10, 4.5))
for a, index in zip(ax.flat, wrong[:10]):
    a.imshow(test_raw[index][0], cmap="gray")
    a.set_title("True: " + class_names[true[index]] + "\nPred: " + class_names[pred[index]], fontsize=8)
    a.axis("off")
plt.show()

**Code Explanation:** `confusion_matrix(true, pred)` counts the images for every pair of (true class, predicted class). We draw it as a coloured table with `imshow`. Then we build a list of all the off-diagonal cells (the mistakes), sort it from the biggest to the smallest, and print the five most common confusions. `np.where(pred != true)` gives the positions of all the wrong predictions, and the last chart shows the first 10 of them with their true and predicted labels.

**Interpretation:** In the matrix a strong diagonal means that most predictions are right, and any bright cell outside the diagonal is a common confusion. In the printed list, look at which pairs come first. For this dataset the top pairs are usually among `Shirt`, `T-shirt/top`, `Pullover` and `Coat`, and sometimes `Sneaker`, `Sandal` and `Ankle boot`. These are the classes that look alike, so these mistakes are understandable, and even people disagree on some of these pictures. Look at the wrong images: some are really hard or look like the predicted class, which means that the model is not completely wrong. If the mistakes are mostly between very different classes, it points to a bug in the code or in the data. Possible next steps are a bigger model, more training, or stronger augmentation, but each change should be judged on the validation set, and the test set should be used again only at the end.

# Part 2: A More Challenging Experiment with CIFAR-10

CIFAR-10 has 60,000 colour pictures (32 x 32 pixels) of 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck. There are 50,000 training images and 10,000 test images, with 6,000 images per class in total.

It is harder than Fashion-MNIST for several reasons: the images have colour, the background is different in every picture, the objects have different positions, angles and sizes, and some classes (cat and dog, for example) look very much alike at 32 x 32 pixels. We follow the same 12 steps, and in each topic we explain what is different from Part 1. The functions `evaluate`, `train_model` and `get_predictions` from Part 1 are reused.

## 2.1 Dataset Loading

**Definition:** The same idea as in Part 1: we get the images and labels with `torchvision.datasets`. This time the class is `datasets.CIFAR10`.

**Example:** `datasets.CIFAR10(root="data", train=True, download=True)` downloads the 50,000 training images (about 170 MB the first time).

**Why it is used?** We use the official training and test split, so that the result can be compared with the results in books and papers.

In [ ]:
cifar_train_raw = datasets.CIFAR10(root="data", train=True, download=True)
cifar_test_raw = datasets.CIFAR10(root="data", train=False, download=True)
cifar_names = cifar_train_raw.classes
print("Training images:", len(cifar_train_raw))
print("Test images:", len(cifar_test_raw))
print("Classes:", cifar_names)

**Code Explanation:** The code is the same as in Part 1 with a different dataset class. The first run downloads and unpacks the data into the `data` folder, and the next runs use the saved copy.

**Interpretation:** You should see 50000 training images, 10000 test images and the 10 class names. If the download fails, check your internet connection. Because the data is saved in the `data` folder, the download is needed only once.

## 2.2 Dataset Inspection

**Definition:** Again we check the size, shape, data type, pixel range and class counts, now for colour images.

**Example:** One CIFAR-10 image has the shape (32, 32, 3): 32 rows, 32 columns and 3 colour channels (red, green, blue).

**Why it is used?** The input layer of the CNN must have 3 input channels this time, and we must compute the mean and the std for each colour channel. Inspection confirms these facts.

In [ ]:
image, label = cifar_train_raw[0]
print("One sample type:", type(image).__name__, "| image size:", image.size, "| mode:", image.mode)
print("Label of the first image:", label, "->", cifar_names[label])
print("Data array shape:", tuple(cifar_train_raw.data.shape))
print("Data type:", cifar_train_raw.data.dtype)
print("Pixel value range:", cifar_train_raw.data.min().item(), "to", cifar_train_raw.data.max().item())
cifar_labels = torch.as_tensor(cifar_train_raw.targets).numpy()
print("Images per class:", np.bincount(cifar_labels))

**Code Explanation:** The same checks as in Part 1. In CIFAR-10 the data array is a NumPy array with the channels last, `(50000, 32, 32, 3)`.

**Interpretation:** The mode `RGB` and the shape `(50000, 32, 32, 3)` confirm 3-channel colour images of 32 x 32 pixels. The pixels are `uint8` from 0 to 255 as before. The class counts are all 5000, so CIFAR-10 is perfectly balanced as well. The first convolution layer will need `in_channels=3`, and `ToTensor()` will move the channels to the front to give (3, 32, 32).

## 2.3 Class Analysis

**Definition:** The same as in Part 1: checking the class balance and thinking about which classes look alike.

**Example:** `cat` and `dog` are both furry animals with four legs, and at 32 x 32 pixels the details that separate them (nose, ears, fur) are only a few pixels. `automobile` and `truck` are both road vehicles.

**Why it is used?** Knowing in advance which classes are close tells us where the model will probably fail, so we can compare it with the confusion matrix later.

**Interpretation:** CIFAR-10 is balanced: 5,000 training images and 1,000 test images per class, so accuracy is a fair metric and no class weighting is needed. Balanced does not mean equally easy. We can divide the classes into two groups: vehicles (airplane, automobile, ship, truck), which have clear shapes and often a recognizable background such as sky or water, and animals (bird, cat, deer, dog, frog, horse), which vary a lot in pose and look more alike. We expect the model to make most of its mistakes among the animal classes, with `cat` and `dog` as the typical hard pair, and some between `automobile` and `truck`. We check this in the error analysis.

## 2.4 Image Visualization

**Definition:** Showing real CIFAR-10 images with their labels.

**Example:** The first training image of each class, in a grid of 2 rows and 5 columns.

**Why it is used?** CIFAR-10 pictures are very small and look blurry when enlarged. Seeing them helps us understand how little information the model has and why this task is hard.

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(10, 4.5))
for i, a in enumerate(ax.flat):
    index = int(np.where(cifar_labels == i)[0][0])
    a.imshow(cifar_train_raw[index][0])
    a.set_title(cifar_names[i])
    a.axis("off")
plt.show()

**Code Explanation:** The same loop as in Part 1. We do not give `cmap="gray"`, so that `imshow` shows the colours.

**Interpretation:** Compared with Fashion-MNIST, the objects are not always centred, the backgrounds are different, and the pictures are small and blurry. The colour is a useful clue for some classes (blue sky for airplanes, blue water for ships, green for frogs and deer) but it is not reliable, so the network must also learn shapes. Because the position and size of the object change from one picture to another, the augmentation (flip and shift) is more useful here than in Part 1.

## 2.5 Preprocessing

**Definition:** The same steps as in Part 1: `ToTensor()`, `Normalize` and a split of the 50,000 training images into 45,000 for training and 5,000 for validation. The difference is that the mean and the std are calculated for each of the 3 colour channels separately.

**Example:** The mean could be, for example, about 0.49 for the red channel, 0.48 for the green channel and 0.45 for the blue channel. `Normalize` then uses a different pair of numbers for each channel.

**Why it is used?** Each colour channel has its own average brightness and spread, so each is standardized by its own numbers. As before, the numbers come from the training images only.

**Interpretation:** There is no separate code. It is written in the DataLoader topic (2.7). We do not resize the images because they all have the same size. We also keep the colour: converting to grayscale would remove information that helps with classes like ships and frogs. Notice that we calculate the mean and the std from our own training data, and we do not copy the numbers from another dataset.

## 2.6 Augmentation

**Definition:** Random changes of the training images only. For CIFAR-10 we use the two standard augmentations: `RandomHorizontalFlip()` and `RandomCrop(32, padding=4)`, which moves the picture by up to 4 pixels.

**Example:** A horse looking to the left is flipped into a horse looking to the right, and a car shifted a few pixels to the side is still the same car.

**Why it is used?** CIFAR-10 has only 5,000 images per class, and a CNN easily memorizes them. Augmentation is one of the most effective ways to reduce this overfitting, and the flip and the shift match the real variation of objects in photos. A vertical flip is not used because the world is not upside down in photos.

**Interpretation:** The code is in the DataLoader topic (2.7). With augmentation, the training accuracy increases more slowly than without it and stays close to the validation accuracy. Without augmentation, a CNN on CIFAR-10 typically reaches a very high training accuracy while the validation accuracy stays much lower. As an exercise, remove the two augmentation lines, train again, and compare the gap between the training and the validation curves.

## 2.7 DataLoader Creation

**Definition:** Same as in Part 1: we build the training, validation and test datasets with the right transforms, and wrap them in `DataLoader`s that deliver batches.

**Example:** One batch is a tensor of shape (128, 3, 32, 32) with 128 labels.

**Why it is used?** It feeds the data to the model in batches, shuffles the training images every epoch, and applies the random augmentation to each image on the fly.

In [ ]:
indices = torch.randperm(len(cifar_train_raw))
n_val = len(indices) // 10
train_idx, val_idx = indices[:-n_val], indices[-n_val:]
pixels = torch.as_tensor(cifar_train_raw.data)[train_idx].float() / 255
cifar_mean = pixels.mean(dim=(0, 1, 2)).tolist()
cifar_std = pixels.std(dim=(0, 1, 2)).tolist()
print("Channel means:", [round(m, 4) for m in cifar_mean])
print("Channel stds :", [round(s, 4) for s in cifar_std])
cifar_train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_full_train = datasets.CIFAR10(root="data", train=True, transform=cifar_train_tf)
cifar_full_val = datasets.CIFAR10(root="data", train=True, transform=cifar_test_tf)
cifar_test_set = datasets.CIFAR10(root="data", train=False, transform=cifar_test_tf)
cifar_train_set = Subset(cifar_full_train, train_idx)
cifar_val_set = Subset(cifar_full_val, val_idx)
cifar_train_loader = DataLoader(cifar_train_set, batch_size=128, shuffle=True)
cifar_val_loader = DataLoader(cifar_val_set, batch_size=256)
cifar_test_loader = DataLoader(cifar_test_set, batch_size=256)
images, labels = next(iter(cifar_train_loader))
print("Training / validation / test images:", len(cifar_train_set), len(cifar_val_set), len(cifar_test_set))
print("One batch of images:", tuple(images.shape), "| labels:", tuple(labels.shape))

**Code Explanation:** The structure is the same as in Part 1. The differences are: the mean and the std are computed over the three dimensions `(0, 1, 2)` (images, rows, columns), which leaves one value per colour channel, `Normalize` receives the three means and the three stds, and the crop is `RandomCrop(32, padding=4)`. The validation images use the transform without augmentation, and they never overlap with the training images.

**Interpretation:** You should see 45000 training, 5000 validation and 10000 test images, and a batch shape of `(128, 3, 32, 32)`: 128 images, 3 channels, 32 x 32 pixels. The three channel means are around 0.45 to 0.49 and the stds around 0.24 to 0.26, which are the usual values for CIFAR-10. The data is read and augmented on the CPU for every batch, which is a part of the training time.

## 2.8 CNN Architecture

**Definition:** A deeper CNN than in Part 1: three convolution blocks (32, 64 and 128 kernels) with Batch Normalization, ReLU and Max Pooling, then Flatten, a fully connected layer with Dropout, and the output layer with 10 scores.

**Example:** The image (3, 32, 32) becomes 32 maps of 16 x 16, then 64 maps of 8 x 8, then 128 maps of 4 x 4, which Flatten turns into 128 x 4 x 4 = 2048 numbers.

**Why it is used?** The images are harder, so the network needs more convolution layers to build more complex features. We also use stronger dropout (0.4) because CIFAR-10 is more prone to overfitting.

In [ ]:
cifar_model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(128 * 4 * 4, 128), nn.ReLU(), nn.Dropout(0.4),
    nn.Linear(128, 10),
)
x = torch.zeros(1, 3, 32, 32)
for layer in cifar_model.eval():
    x = layer(x)
    print(layer.__class__.__name__.ljust(12), tuple(x.shape))
cifar_params = sum(p.numel() for p in cifar_model.parameters())
print("Learnable numbers:", cifar_params)

**Code Explanation:** The same pattern as in Part 1 with one more convolution block. The first convolution has `3` input channels because the images are colour. After three poolings, the 32 x 32 image is reduced to 4 x 4, so the first fully connected layer has `128 * 4 * 4` inputs.

**Interpretation:** Follow the printed shapes from `(1, 3, 32, 32)` to `(1, 128, 4, 4)`, then `(1, 2048)` and finally `(1, 10)`. The height and width are halved three times (32, 16, 8, 4) while the number of feature maps grows (3, 32, 64, 128). Compare the number of learnable numbers with Part 1. It is similar (even a little smaller), because three poolings make the input of the fully connected layer smaller (2048 numbers instead of 3136). But the model needs much more computation per image, because its convolution layers work on larger, 3-channel images and there are more of them. It is still a small network. Much deeper networks (for example ResNets) reach a higher accuracy on CIFAR-10, but they are slower to train.

## 2.9 Training

**Definition:** The same training procedure as in Part 1, using the same `train_model` function: Adam optimizer, cross entropy loss, validation after each epoch, and the best weights kept.

**Example:** We train for 15 epochs. CIFAR-10 is harder, so it needs more epochs than Fashion-MNIST to learn.

**Why it is used?** It teaches the kernels and the weights to recognize the 10 classes. The loss and accuracy curves show how well the learning goes.

In [ ]:
epochs = 15
start = time.time()
cifar_history = train_model(cifar_model, cifar_train_loader, cifar_val_loader, epochs)
cifar_time = time.time() - start
print("Training time: %.0f seconds" % cifar_time)

**Code Explanation:** We call the same `train_model` function with the CIFAR-10 model and loaders. Only the `epochs` value is different. This is why we wrote the training as a function.

**Interpretation:** This cell takes much longer than the Fashion-MNIST one, especially on a CPU (if it is too slow, reduce `epochs` to 5 or 8). Compare the lines with Part 1: the accuracy usually rises more slowly and ends at a lower level, and the validation loss is usually higher. The accuracy usually keeps improving for many epochs. The gap between the training and the validation accuracy tells how much the model overfits, even with augmentation and dropout.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(cifar_history["train_loss"], label="Training")
ax[0].plot(cifar_history["val_loss"], label="Validation")
ax[0].set_title("Loss")
ax[1].plot(cifar_history["train_acc"], label="Training")
ax[1].plot(cifar_history["val_acc"], label="Validation")
ax[1].set_title("Accuracy")
for a in ax:
    a.set_xlabel("Epoch")
    a.legend()
plt.show()

**Code Explanation:** The same two charts as in Part 1, now for the CIFAR-10 model.

**Interpretation:** Use the same rules as before. Lines that go down (loss) or up (accuracy) and stay close together mean healthy learning. A growing gap between the training and the validation lines means overfitting. If the validation line is still going up at the last epoch, the model has not finished learning and more epochs would help. Compare the shape of these curves with the Fashion-MNIST curves: the CIFAR-10 curves are usually slower, lower and less smooth.

## 2.10 Validation

**Definition:** The same as in Part 1: the 5,000 validation images are used after every epoch to track the progress and to choose the weights of the best epoch.

**Example:** If the validation accuracy is best at epoch 13 and gets a little lower at epochs 14 and 15, the model keeps the weights of epoch 13.

**Why it is used?** On CIFAR-10 overfitting is a bigger risk than on Fashion-MNIST, so a separate validation set that does not use augmentation is more important.

**Interpretation:** There is no extra code. Look at the validation loss together with the validation accuracy. If the validation loss starts to rise while the validation accuracy is still flat or rising, the model is becoming over-confident, which is an early sign of overfitting. The validation set is small (5,000 images), so its accuracy changes by a fraction of a percentage point from epoch to epoch. Do not read too much into tiny differences between two epochs or between two models.

## 2.11 Testing

**Definition:** The final measurement of the CIFAR-10 model on the 10,000 test images, which were never used for training or decisions.

**Example:** The overall test accuracy, and the accuracy, precision and recall of each of the 10 classes.

**Why it is used?** It is the honest estimate of how the model works on new images, and the per-class table shows which classes are easy and which are hard.

In [ ]:
cifar_test_loss, cifar_test_acc = evaluate(cifar_model, cifar_test_loader)
print("Test loss: %.4f | Test accuracy: %.2f percent" % (cifar_test_loss, cifar_test_acc * 100))
cifar_true, cifar_pred = get_predictions(cifar_model, cifar_test_loader)
cifar_per_class = pd.Series(cifar_pred == cifar_true).groupby(cifar_true).mean() * 100
cifar_per_class.index = cifar_names
print(cifar_per_class.round(1).sort_values())
print(classification_report(cifar_true, cifar_pred, target_names=cifar_names, digits=3, zero_division=0))

**Code Explanation:** The same steps as in Part 1, reusing `evaluate` and `get_predictions`, with the CIFAR-10 model and test loader.

**Interpretation:** First compare the test accuracy with the best validation accuracy: they should be close. A well-trained small CNN like this one usually reaches a test accuracy in the range of roughly 75 to 85 percent on CIFAR-10, which is much lower than what it usually gets on Fashion-MNIST (around 90 percent or more). Your number may differ, because it depends on the number of epochs and on the random start. In the per-class table, the weakest classes are usually `cat`, then `bird`, `dog` and `deer`, and the strongest are usually `automobile`, `ship`, `truck` and `horse`. Check this on your own output.

## 2.12 Error Analysis

**Definition:** Studying the mistakes: the confusion matrix, the most common confusions, and examples of wrongly classified images.

**Example:** If many images of `cat` are predicted as `dog` and many `dog` as `cat`, the model cannot separate this pair well.

**Why it is used?** It tells us what the model has learned and what it has not, and it helps to choose the next improvement.

In [ ]:
cifar_matrix = confusion_matrix(cifar_true, cifar_pred)
plt.figure(figsize=(7, 6))
plt.imshow(cifar_matrix, cmap="Blues")
plt.xticks(range(10), cifar_names, rotation=90)
plt.yticks(range(10), cifar_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()
pairs = [(cifar_names[i], cifar_names[j], cifar_matrix[i, j]) for i in range(10) for j in range(10) if i != j]
pairs = sorted(pairs, key=lambda p: p[2], reverse=True)
for true_name, pred_name, count in pairs[:5]:
    print(true_name, "predicted as", pred_name, ":", count, "times")
wrong = np.where(cifar_pred != cifar_true)[0]
print("Wrong predictions:", len(wrong), "out of", len(cifar_true))
fig, ax = plt.subplots(2, 5, figsize=(10, 4.5))
for a, index in zip(ax.flat, wrong[:10]):
    a.imshow(cifar_test_raw[index][0])
    a.set_title("True: " + cifar_names[cifar_true[index]] + "\nPred: " + cifar_names[cifar_pred[index]], fontsize=8)
    a.axis("off")
plt.show()

**Code Explanation:** The same analysis as in Part 1, with the CIFAR-10 names and the colour images. We list the five most common confusions, count the wrong predictions, and show the first 10 wrong images.

**Interpretation:** Read the matrix by rows: the row is the true class, and the bright cells outside the diagonal show where its images go wrong. For CIFAR-10 models, the biggest confusions are usually `cat` with `dog`, then `deer` or `bird` with other animals, and sometimes `automobile` with `truck`, and `airplane` with `ship` or `bird` (a similar blue background). Check whether this is also true in your output. Then look at the wrong images: many are small, blurry, partly hidden, or have an unusual pose, which would be difficult even for a person. The mistakes between the animal classes are larger in number than the mistakes between the vehicle classes, as we expected in the class analysis. Next steps to improve could be training for more epochs, a deeper network with residual connections, stronger augmentation, or a model pretrained on a larger dataset (transfer learning).

# Part 3: Comparing the Two Experiments

## 3.1 Fashion-MNIST vs CIFAR-10

**Definition:** A side-by-side table of the two experiments: the size of the input, the size of the model, the training time and the validation and test accuracy.

**Example:** The table is built from the variables that we saved in the two parts, so it shows the numbers of your own run.

**Why it is used?** It shows how the difficulty of the data, and not only the model, decides the result.

In [ ]:
comparison = pd.DataFrame({
    "Dataset": ["Fashion-MNIST", "CIFAR-10"],
    "Input shape": ["1 x 28 x 28", "3 x 32 x 32"],
    "Epochs": [len(fashion_history["val_acc"]), len(cifar_history["val_acc"])],
    "Parameters": [fashion_params, cifar_params],
    "Training time (s)": [round(fashion_time), round(cifar_time)],
    "Best validation acc (%)": [round(max(fashion_history["val_acc"]) * 100, 1), round(max(cifar_history["val_acc"]) * 100, 1)],
    "Test acc (%)": [round(fashion_test_acc * 100, 1), round(cifar_test_acc * 100, 1)],
})
print(comparison.to_string(index=False))

**Code Explanation:** We put the saved results of both experiments into one Pandas table and print it. The numbers come from the variables created earlier in the notebook, so nothing is typed by hand.

**Interpretation:** Compare the two rows. CIFAR-10 normally has a clearly lower test accuracy than Fashion-MNIST, although its model does more computation per image and it usually trains for more epochs and for longer. Check the Parameters column: the number of learnable numbers is not what makes it slower, the amount of computation is. This shows that the difficulty of the data is the main reason: colour photos with different backgrounds, positions and sizes carry far more variation than centred grayscale pictures of clothes, and a small CNN cannot learn all of that variation from 45,000 images. Also compare the best validation accuracy with the test accuracy in each row: they should be close, which shows that our validation and test sets were handled correctly.